In [ ]:
from FIT_python.plot_style import apply_style

apply_style()

In [ ]:
# %% [code]
from pathlib import Path
import pandas as pd
import numpy as np
import ast
from tqdm.auto import tqdm
from joblib import Parallel, delayed
from sklearn.model_selection import ParameterSampler

from FIT_python.pipeline_sex import run_pipeline
from FIT_python.config import RESULTS_DATA_DIR

# 1) Prepare (Clean → Split → Summary)
run_pipeline(
    model_keys=["log_reg"],  # Dummy
    impute_method=None,
    outlier_method=None,
    scaler_method=None,
    reduce_pre_method=None,
    reduce_post_method=None,
    fs_method=None,
    fs_k=None,
)

In [ ]:
import shutil
from FIT_python.pipeline_sex.pipeline_wrapper_sex import _cache_dir

# Cache-Verzeichnis komplett entfernen
shutil.rmtree(_cache_dir)
print(f"Cache gelöscht: {_cache_dir}")

In [2]:
# %% [code] Bayesian search using preset keys
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from tqdm.auto import tqdm
from tqdm_joblib import tqdm_joblib
import joblib
import warnings
from sklearn.exceptions import ConvergenceWarning
from skopt import BayesSearchCV
from skopt.space import Categorical

from FIT_python.data_split_and_summary.transform_wrapper import NumericTransformer
from FIT_python.general_pipeline_steps.imputers import IMPUTERS
from FIT_python.general_pipeline_steps.outliers import OUTLIERS
from FIT_python.general_pipeline_steps.feature_scaler_wrapper import SCALER_PRESETS
from FIT_python.general_pipeline_steps.reducers import REDUCERS
from FIT_python.general_pipeline_steps.feature_selection_wrapper import (
    FeatureSelectionTransformer,
)
from FIT_python.general_pipeline_steps.models import MODELS
from FIT_python.pipeline_sex.grouped_metrics import (
    individual_accuracies,
    individual_majority_stats,
)
from FIT_python.config import SPLITS_DIR, RESULTS_DATA_DIR

warnings.filterwarnings(
    "once", category=ConvergenceWarning, message="Objective did not converge.*"
)
warnings.filterwarnings(
    "once", category=UserWarning, message="X does not have valid feature names.*"
)

# ─── Keys for preset dictionaries ──────────────────────────────────────────────
model_keys = [
    "logreg_l2", "logreg_l1", "rf_small", "rf_med", "rf_large",
    "knn_3", "knn_5", "knn_7", "svm_linear", "svm_rbf",
    "xgb_std", "xgb_hist", "lgbm_std", "lgbm_md10", "lda",
]
outlier_keys = list(OUTLIERS.keys())
reducer_keys = list(REDUCERS.keys())

# ─── Bayesian search space ─────────────────────────────────────────────────────
search_spaces = {
    "impute": Categorical(list(IMPUTERS.values()) + ["passthrough"], transform="identity"),
    "outlier": Categorical(["passthrough"] + [OUTLIERS[k] for k in outlier_keys], transform="identity"),
    "scale": Categorical(list(SCALER_PRESETS.values()) + ["passthrough"], transform="identity"),
    "select__method": Categorical([None, "forward", "lasso", "variance", "random_forest"]),
    "select__k": Categorical([1, 2, 3, 4, 5, 6, 10, 20, 50, 100]),
    "reduce_pre": Categorical(["passthrough"] + [REDUCERS[k] for k in reducer_keys], transform="identity"),
    "reduce_post": Categorical(["passthrough"] + [REDUCERS[k] for k in reducer_keys], transform="identity"),
    "clf": Categorical([MODELS[k] for k in model_keys], transform="identity"),
}

pipeline_order = [
    "impute",
    "outlier",
    "scale",
    "select__method",
    "select__k",
    "reduce_pre",
    "reduce_post",
    "clf",
]

# Reverse lookups for readable IDs
rev_maps = {
    "impute": {v: k for k, v in IMPUTERS.items()},
    "outlier": {v: k for k, v in OUTLIERS.items()},
    "scale": {v: k for k, v in SCALER_PRESETS.items()},
    "reduce_pre": {v: k for k, v in REDUCERS.items()},
    "reduce_post": {v: k for k, v in REDUCERS.items()},
    "clf": {v: k for k, v in MODELS.items()},
}

raw_records, best_records = [], []

# ─── Output directories and scoring setup ──────────────────────────────────────
base_dir = Path(RESULTS_DATA_DIR) / "bayes_search_standard_metrics"
base_dir.mkdir(parents=True, exist_ok=True)
metrics = [
    "maj_test_pct",
    "balanced_test_acc",
    "accuracy_test",
    "mean_test_neg_log_loss",
]
for m in metrics:
    (base_dir / f"best_{m}").mkdir(exist_ok=True)
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "neg_log_loss": "neg_log_loss",
}

# ─── Loop over species ─────────────────────────────────────────────────────────
for species_dir in tqdm(sorted(Path(SPLITS_DIR).iterdir()), desc="Species"):
    if not species_dir.is_dir():
        continue
    species = species_dir.name
    print(f"\n🔄 Processing species: {species}")

    df_train = (
        pd.read_parquet(species_dir / "train.parquet")
        .query("sex in ['f','m']")
        .drop(columns=["Fold"], errors="ignore")
    )
    df_test = (
        pd.read_parquet(species_dir / "test.parquet")
        .query("sex in ['f','m']")
        .drop(columns=["Fold"], errors="ignore")
    )

    if species == "eurasian_otter":
        feature_cols = [
            c
            for c in df_train.columns
            if c.startswith(("dist", "ang", "t"))
            and pd.api.types.is_numeric_dtype(df_train[c])
        ]
    else:
        feature_cols = [
            c
            for c in df_train.columns[5:]
            if pd.api.types.is_numeric_dtype(df_train[c])
        ]
    print(f"  → {len(feature_cols)} numeric features")

    X_tr, y_tr, ids_tr = (
        df_train[feature_cols],
        df_train["sex"].map({"f": 0, "m": 1}).values,
        df_train["individual_id"].values,
    )
    X_te, y_te, ids_te = (
        df_test[feature_cols],
        df_test["sex"].map({"f": 0, "m": 1}).values,
        df_test["individual_id"].values,
    )

    steps = [
        ("transform", NumericTransformer()),
        ("impute", IMPUTERS["rf_default"]),
        ("outlier", OUTLIERS["clip_98"]),
        ("scale", SCALER_PRESETS["standard"]),
        ("reduce_pre", REDUCERS["pca_10"]),
        ("select", FeatureSelectionTransformer(method="forward", k=1)),
        ("reduce_post", REDUCERS["pca_10"]),
        ("clf", MODELS["rf_small"]),
    ]
    pipe = Pipeline(steps)

    bayes = BayesSearchCV(
        estimator=pipe,
        search_spaces=search_spaces,
        n_iter=30,
        scoring="neg_log_loss",
        cv=2, #hier unbedingt noch folds aus dem datenatz nehmen!
        n_jobs=-1,
        random_state=42,
        verbose=1,
        refit=False,
    )

    folds = bayes.cv if isinstance(bayes.cv, int) else getattr(bayes.cv, "n_splits", len(list(bayes.cv)))
    total_fits = bayes.n_iter * folds
    with tqdm_joblib(tqdm(desc=f"{species} Bayes-CV", total=total_fits, leave=False)):
        bayes.fit(X_tr, y_tr)

    cv_res = bayes.cv_results_
    for i, params in enumerate(cv_res["params"]):
        record = {
            "species": species,
            "mean_test_accuracy": cv_res["mean_test_accuracy"][i],
            "mean_test_balanced_accuracy": cv_res["mean_test_balanced_accuracy"][i],
            "mean_test_neg_log_loss": cv_res["mean_test_neg_log_loss"][i],
            **params,
        }
        mdl = clone(pipe).set_params(**params).fit(X_tr, y_tr)
        y_tr_pred = mdl.predict(X_tr)
        y_te_pred = mdl.predict(X_te)
        y_all_true = np.concatenate([y_tr, y_te])
        y_all_pred = np.concatenate([y_tr_pred, y_te_pred])
        ids_all = np.concatenate([ids_tr, ids_te])

        fem_tr, mal_tr, bal_tr = individual_accuracies(y_tr, y_tr_pred, ids_tr)
        fem_te, mal_te, bal_te = individual_accuracies(y_te, y_te_pred, ids_te)
        ct_tr, wr_tr, pct_tr = individual_majority_stats(y_tr, y_tr_pred, ids_tr)
        ct_te, wr_te, pct_te = individual_majority_stats(y_te, y_te_pred, ids_te)
        fem_all, mal_all, bal_all = individual_accuracies(
            y_all_true, y_all_pred, ids_all
        )
        ct_all, wr_all, pct_all = individual_majority_stats(
            y_all_true, y_all_pred, ids_all
        )

        acc_tr = accuracy_score(y_tr, y_tr_pred)
        bal_tr = balanced_accuracy_score(y_tr, y_tr_pred)
        acc_te = accuracy_score(y_te, y_te_pred)
        bal_te = balanced_accuracy_score(y_te, y_te_pred)
        acc_all = accuracy_score(y_all_true, y_all_pred)
        bal_all = balanced_accuracy_score(y_all_true, y_all_pred)

        record.update(
            dict(
                female_train_acc=fem_tr,
                male_train_acc=mal_tr,
                balanced_train_acc=bal_tr,
                accuracy_train=acc_tr,
                female_test_acc=fem_te,
                male_test_acc=mal_te,
                balanced_test_acc=bal_te,
                accuracy_test=acc_te,
                female_full_acc=fem_all,
                male_full_acc=mal_all,
                balanced_full_acc=bal_all,
                accuracy_full=acc_all,
                maj_train_count=ct_tr,
                maj_train_wrong=wr_tr,
                maj_train_pct=pct_tr,
                maj_test_count=ct_te,
                maj_test_wrong=wr_te,
                maj_test_pct=pct_te,
                maj_full_count=ct_all,
                maj_full_wrong=wr_all,
                maj_full_pct=pct_all,
            )
        )

        pid_parts = []
        for key in pipeline_order:
            val = params.get(key)
            if key in rev_maps and val in rev_maps[key]:
                val = rev_maps[key][val]
            pid_parts.append(f"{key}={val if val is not None else 'None'}")
        record["pipeline_id"] = ";".join(pid_parts)
        for k, v in record.items():
            if pd.isna(v):
                record[k] = "None"
        raw_records.append(record)

    df_eval = pd.DataFrame([r for r in raw_records if r["species"] == species])
    for metric in metrics:
        best_idx = df_eval[metric].idxmax()
        best_params = cv_res["params"][best_idx]
        best_record = df_eval.loc[best_idx].to_dict()
        best_record["best_metric"] = metric

        best_pipe = clone(pipe).set_params(**best_params).fit(X_tr, y_tr)
        out_path = base_dir / f"best_{metric}" / f"{species}.joblib"
        joblib.dump(best_pipe, out_path)
        print(f"✅ Saved model for '{species}' ({metric}) at:\n   {out_path}")
        best_records.append(best_record)

    df_all = pd.DataFrame(raw_records).fillna("None")
    df_best = pd.DataFrame(best_records).fillna("None")
    df_all.to_csv(base_dir / "all_results.csv", index=False)
    df_best.to_csv(base_dir / "best_models.csv", index=False)


Species:   0%|          | 0/10 [00:00<?, ?it/s]


🔄 Processing species: aj_someringii
  → 136 numeric features


aj_someringii Bayes-CV:   0%|          | 0/60 [00:00<?, ?it/s]

  0%|          | 0/60 [00:00<?, ?it/s]

Fitting 2 folds for each of 1 candidates, totalling 2 fits
Fitting 2 folds for each of 1 candidates, totalling 2 fits
Fitting 2 folds for each of 1 candidates, totalling 2 fits
Fitting 2 folds for each of 1 candidates, totalling 2 fits
Fitting 2 folds for each of 1 candidates, totalling 2 fits
Fitting 2 folds for each of 1 candidates, totalling 2 fits
Fitting 2 folds for each of 1 candidates, totalling 2 fits
Fitting 2 folds for each of 1 candidates, totalling 2 fits
Fitting 2 folds for each of 1 candidates, totalling 2 fits


ValueError: X has 136 features, but PCA is expecting 1 features as input.